# Status report

By Ben Welsh

Generates basic statistics from [News Homepages database extracts](https://palewi.re/docs/news-homepages/extracts.html).

In [90]:
import json
import pandas as pd
import altair as alt
from pathlib import Path
from datetime import datetime, timedelta

In [91]:
alt.renderers.enable('altair_saver', fmts=['vega-lite', 'png'])

RendererRegistry.enable('altair_saver')

In [92]:
this_dir = Path("__file__").parent.absolute()

In [93]:
sources_dir = this_dir.parent / "newshomepages" / "sources"

In [94]:
extracts_dir = this_dir.parent / "extracts" / "csv"

In [95]:
df = pd.read_csv(
    extracts_dir / "screenshot-files.csv",
    parse_dates=["mtime"],
    usecols=["identifier", "handle", "file_name", "mtime"]
)

In [96]:
df['date'] = df.mtime.dt.date

In [97]:
df["date"] = pd.to_datetime(df["date"])

How many total sites?

In [98]:
total_sites = len(df.handle.unique())

In [99]:
total_sites

221

How many total screenshots?

In [100]:
total_screenshots = len(df)

In [101]:
total_screenshots

23318

When did we start?

In [102]:
start_date = min(df.date)

In [103]:
start_date

Timestamp('2022-03-22 00:00:00')

How many screenshots in the last week?

In [104]:
today = datetime.now().date()

In [105]:
today

datetime.date(2022, 7, 2)

In [106]:
one_week_ago = today - timedelta(days=7)

In [107]:
one_week_ago

datetime.date(2022, 6, 25)

In [108]:
df_this_week = df[df.date > pd.to_datetime(one_week_ago)]

In [109]:
screenshots_this_week = len(df_this_week)

In [110]:
screenshots_this_week

5076

Write out data points

In [111]:
output = dict(
    total_sites=total_sites,
    total_screenshots=total_screenshots,
    screenshots_this_week=screenshots_this_week,
)

In [112]:
json.dump(output, open(this_dir / 'status-report.json', 'w'), indent=2)

Chart the number of sites by date

In [113]:
sites_by_date = df[['date', 'handle']].drop_duplicates().groupby("date").size().rename("sites").reset_index().sort_values("date")

In [114]:
sites_by_date['rolling_mean'] = sites_by_date.sites.rolling(7).mean()

In [119]:
chart = alt.Chart(
    sites_by_date,
    title="Sites archived by @newshomepages",
    width=500
)

bars = chart.mark_bar(
    fill="#cecece"
).encode(
    x=alt.X("date:T", title="Date", timeUnit="yearmonthdate", axis=alt.Axis(format="%B %d", grid=False)),
    y=alt.Y("sites:Q", title="Sites"),
)

line = chart.mark_line(color='#727272', strokeWidth=3).encode(
    x='date:T',
    y='rolling_mean:Q'
)

label = chart.encode(
    x=alt.X('max(date):T'),
    y=alt.Y('rolling_mean:Q', aggregate=alt.ArgmaxDef(argmax='date')),
    text='rolling_mean'
)

# Create a text label
text = label.mark_text(align='left', dx=4)

# Create a circle annotation
circle = label.mark_circle(size=75, color="#727272")

(bars + line + circle) #.save(this_dir / 'sites-by-date.svg')

AttributeError: 'WebDriver' object has no attribute 'find_element_by_id'

alt.LayerChart(...)

Chart the number of screenshots by date

In [116]:
screenshots_by_date = df.groupby("date").size().rename("screenshots").reset_index().sort_values("date")

In [117]:
screenshots_by_date['rolling_mean'] = screenshots_by_date.screenshots.rolling(7).mean()

In [118]:
chart = alt.Chart(
    screenshots_by_date,
    title="Screenshots saved by @newshomepages",
    width=500
)

bars = chart.mark_bar(
    fill="#cecece"
).encode(
    x=alt.X("date:T", title="Date", timeUnit="yearmonthdate", axis=alt.Axis(format="%B %d", grid=False)),
    y=alt.Y("screenshots:Q", title="Screenshots"),
)

line = chart.mark_line(color='#727272', strokeWidth=3).encode(
    x='date:T',
    y='rolling_mean:Q'
)

label = chart.encode(
    x=alt.X('max(date):T'),
    y=alt.Y('rolling_mean:Q', aggregate=alt.ArgmaxDef(argmax='date')),
    text='rolling_mean'
)

# Create a text label
text = label.mark_text(align='left', dx=4)

# Create a circle annotation
circle = label.mark_circle(size=75, color="#727272")

(bars + line + circle).save(this_dir / 'screenshots-by-date.png')

SessionNotCreatedException: Message: session not created: This version of ChromeDriver only supports Chrome version 92
Current browser version is 103.0.5060.53 with binary path /usr/bin/google-chrome
Stacktrace:
#0 0x55e6a1da4a63 <unknown>
#1 0x55e6a1b19458 <unknown>
#2 0x55e6a1b3fe84 <unknown>
#3 0x55e6a1b3b6c3 <unknown>
#4 0x55e6a1b37e9f <unknown>
#5 0x55e6a1b71b72 <unknown>
#6 0x55e6a1b6bfd3 <unknown>
#7 0x55e6a1b42514 <unknown>
#8 0x55e6a1b43505 <unknown>
#9 0x55e6a1dd0e2e <unknown>
#10 0x55e6a1de6886 <unknown>
#11 0x55e6a1dd1d75 <unknown>
#12 0x55e6a1de7d94 <unknown>
#13 0x55e6a1dc88eb <unknown>
#14 0x55e6a1e02cd8 <unknown>
#15 0x55e6a1e02e58 <unknown>
#16 0x55e6a1e1cdfd <unknown>
#17 0x7fbca6177b43 <unknown>
